In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tifffile as tf
from tifffile import imread
from tifffile import imwrite

from tqdm import tqdm

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
# Provide path to input files

images_dir = Path(r'input_path')
images_path = os.path.join(images_dir,'*.stk') 
images_files = np.sort(glob.glob(images_path))

print(images_dir)
print(images_path)
print(len(images_files))

In [ ]:
# Split images into separate channels

images_files_halo = [f for f in images_files if 'conf561' in f]
images_files_tubulin = [f for f in images_files if 'conf488' in f]
images_files_actin = [f for f in images_files if 'conf640' in f]

In [ ]:
# Initialize lists to store images by channel

images_tubulin = []
images_actin = []

# Sort images into respective lists based on filename cues
for file in tqdm(images_files):
    if 'conf488' in file:
        images_tubulin.append(imread(file))
    elif 'conf640' in file:
        images_actin.append(imread(file))

# Check the lists
print(f"Tubulin images: {len(images_tubulin)}")
print(f"Actin images: {len(images_actin)}")

In [ ]:
np.unique(images_actin[0])

In [ ]:
# Function to extract the center slice
def extract_center_slice(image):
    """
    Extract the center slice of a 3D image stack.

    Parameters:
    image (numpy.ndarray): A 3D numpy array representing the image stack 
                           (shape: slices x height x width).

    Returns:
    numpy.ndarray: The center slice (2D array).
    """
    center_index = image.shape[0] // 2  # Calculate the center slice index
    return image[center_index]


# Apply max projection to each list
center_slice_tubulin = [extract_center_slice(img) for img in images_tubulin]
center_slice_actin = [extract_center_slice(img) for img in images_actin]

# Check the number of max projections created
print(f"Single slices for Tubulin: {len(center_slice_tubulin)}")
print(f"Single slices for Actin: {len(center_slice_actin)}")

In [ ]:
# Plot whole cell and nucleus mask side by side

fig, ax = plt.subplots(1, 2, figsize = (8,4))
ax[0].imshow(center_slice_tubulin[0])
ax[1].imshow(center_slice_actin[0])

In [ ]:
# Provide path to output directory

single_slice_output_dir = r'output_path'

In [ ]:
# Save max projections to seperate folders

# Define the folder names
folders = ['Tubulin', 'Actin']

# Create the folders if they don't exist
for folder in folders:
    folder_path = os.path.join(single_slice_output_dir, folder)
    os.makedirs(folder_path, exist_ok=True)

# Function to save a max projection image
def save_max_projection_image(image, original_file, channel_name):
    # Extract the file name (without path)
    file_name = os.path.basename(original_file)
    
    # Extract the base name (remove everything after the last '_')
    base_name = file_name.rsplit('_', 1)[0]  # This splits at the last underscore
    
    # Create the new filename with the appropriate suffix (e.g., "_tubulin", "_actin")
    new_file_name = base_name + f"_{channel_name}.tif"
    
    # Define the output path
    output_file = os.path.join(single_slice_dir, channel_name, new_file_name)
    
    # Check if file already exists to prevent overwriting
    if os.path.exists(output_file):
        print(f"Warning: {output_file} already exists! Skipping...")
        return
    
    # Save the max projection image to the corresponding folder
    tf.imwrite(output_file, image)
    print(f"Saved {output_file}")

# Now save the max projections for each channel

for i, image_file in enumerate(images_files_tubulin):
    save_max_projection_image(center_slice_tubulin[i], image_file, 'tubulin')

for i, image_file in enumerate(images_files_actin):
    save_max_projection_image(center_slice_actin[i], image_file, 'actin')
